In [ ]:
pip install datasets

In [ ]:
pip install scikit-learn

In [ ]:
pip install torch

In [ ]:
from datasets import load_dataset
dataset = load_dataset("DeepPavlov/snips")
print(dataset)

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame(dataset['train'])
print(df['label'].unique())
print(df['label'].value_counts())

In [ ]:
print(dataset['train'].features)
print()
# See a few examples
for i in range(5):
    print(dataset['train'][i])

In [ ]:
label_map = {
    0: "AddToPlaylist",
    1: "BookRestaurant", 
    2: "GetWeather",
    3: "PlayMusic",
    4: "RateBook",
    5: "SearchCreativeWork",
    6: "SearchScreeningEvent"
}

# Verify distribution
for k, v in label_map.items():
    count = len(df[df['label'] == k])
    print(f"{v}: {count}")

In [ ]:
from collections import Counter

In [ ]:
# Tokenize all utterances
def tokenize(sentence):
    return sentence.lower().split()
    
# Build vocabulary
counter = Counter()
for sample in dataset['train']:
    counter.update(tokenize(sample['utterance']))

# Create word to index mapping
# 0 = padding, 1 = unknown
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word, freq in counter.items():
    if freq >= 2:  # ignore words appearing only once
        word2idx[word] = len(word2idx)

print(f"Vocabulary size: {len(word2idx)}")

# Dataset Splitting

In [ ]:
from sklearn.model_selection import train_test_split

The dataset already comes pre-split into train and test from HuggingFace.  
However, we do need a validation set, therefore, we'll split dataset['train'] to get validation set(only 10%).

In [ ]:
train_data, val_data = train_test_split(
    dataset['train'].to_pandas(), 
    test_size=0.1,      # 10% for validation
    random_state=100,
    stratify=dataset['train']['label']  # keeps class balance
)
print(f"Train: {len(train_data)}")
print(f"Val: {len(val_data)}")
print(f"Test: {len(dataset['test'])}")

# Dataset Encoder

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
MAX_LEN = 50

def encode(sentence):
    tokens = tokenize(sentence)
    ids = [word2idx.get(t, 1) for t in tokens]  # 1 = <UNK>
    # pad or truncate to MAX_LEN
    ids = ids[:MAX_LEN] + [0] * (MAX_LEN - len(ids))
    return ids

class SNIPSDataset(Dataset):
    def __init__(self, data):
        self.data = data
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx] if hasattr(self.data, 'iloc') else self.data[idx]
        x = torch.tensor(encode(row['utterance']), dtype=torch.long)
        y = torch.tensor(row['label'], dtype=torch.long)
        return x, y

train_dataset = SNIPSDataset(train_data)
val_dataset = SNIPSDataset(val_data)
test_dataset = SNIPSDataset(dataset['test'])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
pip install pytorch-crf

# Model

In [ ]:
import torch
from TorchCRF import CRF

In [ ]:
import torch.nn as nn
from TorchCRF import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(BiLSTM_CRF, self).__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # BiLSTM layer
        self.bilstm = nn.LSTM(
            embedding_dim,
            hidden_dim // 2,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_dim, num_classes)
        
        # CRF layer
        self.crf = CRF(num_classes)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.embedding(x)           # (batch_size, seq_len, embedding_dim)
        lstm_out, _ = self.bilstm(embedded)    # (batch_size, seq_len, hidden_dim)
        emissions = self.fc(lstm_out)          # (batch_size, seq_len, num_classes)
        return emissions

# Initialize model
VOCAB_SIZE = len(word2idx)
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_CLASSES = 7

model = BiLSTM_CRF(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, NUM_CLASSES)
print(model)

# Optimizer

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    for x, y in loader:
        optimizer.zero_grad()
        emissions = model(x)                  # (batch, seq_len, num_classes)
        loss = -model.crf(emissions, y.unsqueeze(1).expand_as(emissions[:,:,0]))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

print("Training setup ready")

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            emissions = model(x)
            predictions = model.crf.decode(emissions)
            # take first token prediction as intent label
            preds = torch.tensor(predictions[0])
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer)
    val_acc = evaluate(model, val_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

In [ ]:
test_acc = evaluate(model, test_loader)
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
torch.save(model.state_dict(), 'module1_bilstm_crf.pth')
print("Model saved")

In [ ]:
def predict_intent(sentence, model, word2idx, label_map):
    model.eval()
    tokens = sentence.lower().split()
    ids = [word2idx.get(t, 1) for t in tokens]
    ids = ids[:MAX_LEN] + [0] * (MAX_LEN - len(ids))
    x = torch.tensor([ids], dtype=torch.long)
    with torch.no_grad():
        emissions = model(x)
        predictions = model.crf.decode(emissions)
    intent_idx = predictions[0][0]
    return label_map[intent_idx]

# Test it
label_map = {
    0: "AddToPlaylist",
    1: "BookRestaurant",
    2: "GetWeather",
    3: "PlayMusic",
    4: "RateBook",
    5: "SearchCreativeWork",
    6: "SearchScreeningEvent"
}

print(predict_intent("play some jazz music", model, word2idx, label_map))
print(predict_intent("what is the weather in london", model, word2idx, label_map))
print(predict_intent("book a table for two", model, word2idx, label_map))